In [ ]:
import re
from _collections import defaultdict

In [10]:
# Convert all unique words into format: hello -> _ h e l l o
def initialize_vocabulary(corpus):
    # Init vocabulary with dict has default value = 0
    vocabulary = defaultdict(int)
    # Init charset
    charset = set()

    for word in corpus:
        # Add '_' to begin of word: hello -> _hello
        word_with_marker = '_' + word
        # Split word to characters
        characters = list(word_with_marker)
        # Update unique char to charset
        charset.update(characters)
        # [_,h,e,l,l,o] -> _ h e l l o
        tokenized_word = ' '.join(characters)
        # Update dict value
        vocabulary[tokenized_word] += 1
    return vocabulary, charset

In [45]:
vocab, charset = initialize_vocabulary(["_Let", "'", "s", "_proceed", "_to", "_the", "_language", "_model", "ing", "_part", "."])
print(vocab)
print()
print(charset)

defaultdict(<class 'int'>, {'_ _ L e t': 1, "_ '": 1, '_ s': 1, '_ _ p r o c e e d': 1, '_ _ t o': 1, '_ _ t h e': 1, '_ _ l a n g u a g e': 1, '_ _ m o d e l': 1, '_ i n g': 1, '_ _ p a r t': 1, '_ .': 1})

{'n', 'e', 'o', "'", 'c', 'm', 'a', 'L', 'r', '.', 'h', 'l', 'p', 'g', 's', 'd', 'u', '_', 'i', 't'}


In [12]:
def get_pair_counts(vocabulary):
    # Init pair_counts with dict has default value = 0
    pair_counts = defaultdict(int)

    for tokenized_word, count in vocabulary.items():
        # Split each tokenized word to characters by space
        tokens = tokenized_word.split()
        for i in range(len(tokens) -1):
            pair = (tokens[i], tokens[i+1])
            pair_counts[pair] += count
    return  pair_counts

In [13]:
pair_counts = get_pair_counts(vocab)
print(pair_counts)

defaultdict(<class 'int'>, {('_', 'h'): 4, ('h', 'e'): 3, ('e', 'l'): 3, ('l', 'l'): 2, ('l', 'o'): 1, ('h', 'i'): 1, ('_', 'l'): 1, ('l', 'i'): 1, ('i', 'g'): 1, ('g', 'h'): 1, ('h', 't'): 1, ('t', 'n'): 1, ('n', 'i'): 1, ('i', 'n'): 1, ('n', 'g'): 1, ('l', 'p'): 1})


In [43]:
def merge_pair(vocabulary, pair):
    new_vocabulary = {}
    # Join the pair and keep special char
    bigram = re.escape(' '.join(pair))
    print('bi:',bigram)
    # Ensure bigram not a part of a bigger word
    pattern = re.compile(r"(?<!\s)" + bigram + r"(?!\s)")
    print('pat:',pattern)
    for tokenized_word, count in vocabulary.items():
        # Replace tokenized word with bigram: _ h e l l o, ('_','h') -> _h e l l o
        print(tokenized_word)
        new_tokenized_word = pattern.sub("".join(pair), tokenized_word)
        print(new_tokenized_word)

        new_vocabulary[new_tokenized_word] = count
    return new_vocabulary

In [44]:
new_vocab = merge_pair({'_ h e l l o': 1, '_ h i': 1, '_ h e l l': 1, '_ l i g h t n i n g': 1, '_ h e l p': 1}, ('_', 'h'))
print(new_vocab)

bi: _ h
pat: re.compile('(?<!\\s)_ h(?!\\s)')
_ h e l l o
_ h e l l o
_ h i
_ h i
_ h e l l
_ h e l l
_ l i g h t n i n g
_ l i g h t n i n g
_ h e l p
_ h e l p
{'_ h e l l o': 1, '_ h i': 1, '_ h e l l': 1, '_ l i g h t n i n g': 1, '_ h e l p': 1}


In [30]:
def byte_pair_encoding(corpus, vocab_size):
    vocabulary, charset = initialize_vocabulary(corpus)
    merges = []
    tokens = set(charset)
    while len(tokens) < vocab_size:
        pair_counts = get_pair_counts(vocabulary)
        if not pair_counts:
            break
        most_frequent_pair = max(pair_counts, key=pair_counts.get)
        print((most_frequent_pair))
        merges.append(most_frequent_pair)
        vocabulary = merge_pair(vocabulary, most_frequent_pair)
        print(vocabulary)
        new_token = ''.join(most_frequent_pair)
        tokens.add(new_token)

    return vocabulary, merges, charset, tokens

In [31]:
vocab, merges, charset, tokens = byte_pair_encoding(['hello', 'hi', 'hell', 'lightning', 'help'], 11)

('_', 'h')
{'_ h e l l o': 1, '_ h i': 1, '_ h e l l': 1, '_ l i g h t n i n g': 1, '_ h e l p': 1}


In [ ]:
def tokenize_word(word, merges, vocabulary, charset, unk_token="<UNK>"):
    word = '_' + word
    if word in vocabulary:
        return [word]
    tokens = [char if char in charset else unk_token for char in word]
    for left, right in merges:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i:i+2] == [left, right]:
                tokens[i:i+2] = [left + right]
            else:
                i += 1
    return tokens